## Data Preprocessing Phase

### Data Loader

In [ ]:
from abc import ABC
from enum import Enum
from pathlib import Path
from pandas import DataFrame
import pandas as pd


class FileTypes(Enum):
    LINK = "http//"
    XML = ".xml"
    CSV = ".csv"


class FileTypeChecker(ABC):
    def __init__(self, file_path: str | Path):
        self.file_path = str(file_path)

    def checker(self):
        if self.file_path.endswith(".csv") == FileTypes.CSV:
            return self.file_path
        elif self.file_path.endswith(".xml") == FileTypes.XML:
            return self.file_path
        else:
            raise FileNotFoundError("File not found!")


class DataLoader:
    def __init__(self, file_path: str | Path):
        self.file_path = file_path
        self.file = FileTypeChecker(self.file_path).checker()

    def loader(self) -> DataFrame:
        return pd.read_csv(self.file)

    def get_values(self, df: DataFrame):
        return df.shape


### Preprocessing Section

In [ ]:
from pandas import DataFrame


class PreProcessing:
    def __init__(self, df: DataFrame) -> None:
        self.df = df

    def remove_duplicate(self) -> DataFrame:
        return self.df.drop_duplicates()

    def fill_nan(self):
        return self.df.fillna()

    def feature_extract(self, column: list):
        return self.df.drop(column, axis=1)

    def target_extract(self, lable: str):
        return self.df[lable]


## Training Phase

### Setup Model For Training

In [ ]:
from typing import Any, Dict

from sklearn.ensemble import RandomForestClassifier


class SetupModel:
    def __init__(self, params: Dict[str, Any]) -> None:
        self.params = params

    def setup_model(self) -> RandomForestClassifier:
        return RandomForestClassifier(
            n_estimators=self.params["n_estimators"],
            random_state=self.params["random_state"],
            n_jobs=self.params["n_jobs"],
            max_depth=self.params["max_depth"],
        )


### Model Training for Make Prediction

In [ ]:
class TrainModel:
    def __init__(self, model: RandomForestClassifier) -> None:
        self.model = model

    def train(self, X_train, y_train) -> RandomForestClassifier:
        return self.model.fit(X_train, y_train)


## Evaluation Phase

### Get Predicted Artifacts

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score


class PreEvaluation:
    def __init__(self, model: RandomForestClassifier) -> None:
        self.model = model

    def predict_(self, X_test: DataFrame):
        return self.model.predict_proba(X_test)

    def accuracy(self, X_test, pred):
        return accuracy_score(X_test, pred)

    def confusion_matrix(self, y_test, pred):
        cm = confusion_matrix(y_test, pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title("HAR Model - Confusion Matrix", fontsize=16)
        plt.ylabel("Actual Activity")
        plt.xlabel("Predicted Activity")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig("confusion_matrix.png")
        plt.show()
        return cm

    def cross_validation(self, X_train, y_train, X_test, y_test):
        cv_scores = cross_val_score(
            self.model,
            np.vstack([X_train, X_test]),
            np.concatenate([y_train, y_test]),
            cv=5,
            n_jobs=-1,
        )

        print(f"CV Scores: {cv_scores}")
        print(f"Mean Accuracy: {cv_scores.mean()*100:.2f}%")
        print(f"Std: {cv_scores.std()*100:.2f}%")
        print(
            f"Reliable Accuracy: {(cv_scores.mean()-cv_scores.std())*100:.2f}% - {(cv_scores.mean()+cv_scores.std())*100:.2f}%"
        )
        return cv_scores


### Get Predictions and Outcome

In [ ]:
import joblib


class PostEvaluation:
    def __init__(self) -> None:
        pass

    def save_model(self, model, encoder):
        joblib.dump(model, "HAR_RF_MODEL.pkl")
        files.download("HAR_RF_MODEL.pkl")  # Colab Downloader
        joblib.dump(encoder, "HAR_LAEN.pkl")
        files.download("HAR_LAEN.pkl")  # Colab Downloader

    def gen_prediction(self, final_model: RandomForestClassifier, X_value):
        final_model.predict(X_value)


In [ ]:
if __name__ == "__main__":

    params = {
        "n_estimators": 5,
        "max_depth": None,
        "random_state": 42,
        "n_jobs": -1,
    }

    train_file = DataLoader("")
    train_df = train_file.loader()
    print(train_file.get_values(train_df))

    test_file = DataLoader("")
    test_df = test_file.loader()
    print(test_file.get_values(test_df))

    train_pre_df = PreProcessing(train_df)
    X_train = train_pre_df.feature_extract(["any"])
    y_train = train_pre_df.target_extract("any")

    test_pre_df = PreProcessing(train_df)
    X_test = test_pre_df.feature_extract(["any"])
    y_test = test_pre_df.target_extract("any")

    model_setup = SetupModel(params)
    raw_model = model_setup.setup_model()

    initialize_model = TrainModel(raw_model)
    trained_model = initialize_model.train(X_train, y_train)

    evaluate = PreEvaluation(trained_model)
    predict = evaluate.predict_(X_test)
    print(f"Prediction: {predict}")
    accuracy = evaluate.accuracy(X_test, predict)
    print(f"Accuracy: {accuracy}")
    confusion_matrix = evaluate.confusion_matrix(y_test, predict)
    cross_val_score = evaluate.cross_validation(X_train, y_train, X_test, y_test)

    output = PostEvaluation()
    prediction = output.gen_prediction(trained_model, [])
